# LAB Módulo 02

Notebook com três exemplos práticos usando o SDK `openai` com `AzureOpenAI`:

1. **Chat Completion** — chamada simples de conversa
2. **Streaming** — resposta em tempo real (Server-Sent Events)
3. **Geração de imagem** — DALL·E 3

O client é criado **uma única vez** na seção de configuração e reaproveitado nos três exemplos.

## 1. Requisitos

Instale as dependências (rode a célula abaixo apenas na primeira vez).

In [ ]:
%pip install openai python-dotenv requests --quiet

## 2. Configuração das credenciais

Crie um arquivo `.env` na mesma pasta deste notebook com o conteúdo:

```env
AZURE_OPENAI_ENDPOINT=https://SEU-RECURSO.openai.azure.com/
AZURE_OPENAI_KEY=sua-chave-aqui
```

> Os nomes de `model` nas chamadas correspondem aos **nomes dos deployments** no seu recurso Azure OpenAI — ajuste se os seus forem diferentes de `gpt-4o` e `dall-e-3`.

## 3. Criação do client (compartilhado)

Executa uma vez e serve para os três exemplos.

In [ ]:
from openai import AzureOpenAI
from dotenv import load_dotenv
import os

# Carrega variáveis de ambiente (.env)
load_dotenv()

# Cria o client da Azure OpenAI
client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-02-15-preview",  # versão válida que suporta chat e dall-e-3
)

# Verificação rápida da configuração
assert os.getenv("AZURE_OPENAI_ENDPOINT"), "Faltou AZURE_OPENAI_ENDPOINT no .env"
assert os.getenv("AZURE_OPENAI_KEY"), "Faltou AZURE_OPENAI_KEY no .env"
print("Client Azure OpenAI configurado com sucesso.")

## 4. Exemplo 1 — Chat Completion

Chamada básica de conversa. O parâmetro `temperature` controla a criatividade da resposta.

In [ ]:
# Chamada de chat completion
response = client.chat.completions.create(
    model="gpt-4o",  # nome do deployment no Azure
    messages=[
        {"role": "system", "content": "Assistente em PT-BR"},
        {"role": "user", "content": "O que é uma API?"},
    ],
    temperature=0.3,
)

# Exibe resposta
print(response.choices[0].message.content)

## 5. Exemplo 2 — Streaming

Com `stream=True`, a resposta chega em pedaços (chunks) em tempo real, ideal para exibir o texto conforme é gerado.

In [ ]:
# Mesmo client AzureOpenAI da seção de configuração
stream = client.chat.completions.create(
    model="gpt-4o",  # nome do deployment no Azure
    messages=[
        {"role": "user", "content": "Escreva um poema"}
    ],
    stream=True,  # ativa streaming
)

# Consumir resposta em tempo real (Server-Sent Events)
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta and chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)  # sem buffer

## 6. Exemplo 3 — Geração de imagem (DALL·E 3)

Gera uma imagem a partir de um prompt e salva localmente como `output.png`.

- `size`: `1024x1024`, `1792x1024` ou `1024x1792`
- `quality`: `hd` ou `standard`
- `style`: `vivid` ou `natural`

In [ ]:
import requests

# Geração da imagem com DALL-E (Azure OpenAI)
result = client.images.generate(
    model="dall-e-3",  # nome do deployment no Azure
    prompt="Um leão estudando finanças com luz dourada",
    size="1024x1024",  # opções: 1024x1024, 1792x1024, 1024x1792
    quality="hd",      # ou "standard"
    style="vivid",     # "vivid" ou "natural"
    n=1,
)

# URL da imagem gerada
image_url = result.data[0].url

# Download e salvamento local
img = requests.get(image_url).content
with open("output.png", "wb") as f:
    f.write(img)

print("Imagem salva como output.png")

### Visualizar a imagem gerada no notebook

In [ ]:
from IPython.display import Image, display

display(Image(filename="output.png"))